# 02a · Geneformer V2-104M embeddings

**Environment:** `envs/geneformer.yml` (this foundation model has heavy, specific dependencies: run this notebook in its own environment).
**Output:** one `.h5ad` per dataset whose `X` is the frozen embedding (768-d) and whose `obs` carries `cell_id`, `cell_type`, `master_split`.
The backbone is frozen: no fine-tuning, no gradients.

Get the model: `git lfs install && git clone https://huggingface.co/ctheodoris/Geneformer`  (checkpoint folder `Geneformer-V2-104M`).

> These extraction functions are ported from the authors' working notebooks and need the model's own code and checkpoint.
> They are not run in the repository's tests. Try them on the smallest dataset first and check the shapes at the end.

In [ ]:
from pathlib import Path
from heterosc.extraction.geneformer import extract_geneformer_cells, symbol_to_ensembl
import anndata as ad, pandas as pd, numpy as np

DATA = Path("data")                       # from notebook 01
OUT = Path("outputs/embeddings")          # shared layout: {dataset}/{expert}/{dataset}_{expert}_with_embeddings.h5ad
DATASETS = ["Myeloid"]                    # start small; then add Lung, Liver, Pancreas, Skin
GENEFORMER_REPO = Path("checkpoints/Geneformer")

## Part A · annotation datasets

In [ ]:
for ds in DATASETS:
    extract_geneformer_cells(
        DATA / "annotation" / "processed" / f"{ds}_ready_master_split.h5ad",
        OUT / ds / "geneformer" / f"{ds}_geneformer_with_embeddings.h5ad",
        geneformer_repo=GENEFORMER_REPO)

## Part B · cell lines for drug response
Uses `data/drug_response/cellline_expression.h5ad` (CCLE expression, 697 genes). Output layout used by notebook 04:
`outputs/drug_response_embeddings/{expert}/cellline_{expert}_embeddings.h5ad`.

In [ ]:
# Geneformer needs Ensembl ids in var['ensembl_id']; the cell-line file has gene symbols, so map them first
a = ad.read_h5ad(DATA / "drug_response" / "cellline_expression.h5ad")
to_ens = symbol_to_ensembl(GENEFORMER_REPO)
a.var["ensembl_id"] = [to_ens(g) for g in a.var_names]
a.obs["cell_id"] = a.obs_names.astype(str)
a.write_h5ad("outputs/cellline_for_geneformer.h5ad")
extract_geneformer_cells("outputs/cellline_for_geneformer.h5ad",
                         "outputs/drug_response_embeddings/geneformer/cellline_geneformer_embeddings.h5ad",
                         geneformer_repo=GENEFORMER_REPO, extra_obs=())

## Check

In [ ]:
from heterosc.extraction import load_embeddings
import glob
for f in sorted(glob.glob("outputs/**/*_geneformer_*embeddings.h5ad", recursive=True)):
    X, obs = load_embeddings(f)
    print(f, X.shape, "(expected dim 768)")